# Elysium X 150 FR: original synthetic pilot
Status: not trained. 30-conversation smoke pilot, NOT a large dataset. 7B Apache-2.0 local teacher in 4-bit; outputs/labels synthetic. Saved to private Drive after every accepted conversation. Review before scaling or training. No paid API.


In [ ]:
%pip -q install transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.1 peft==0.17.1 datasets==4.1.1


In [ ]:
"""Original synthetic multi-turn pilot. Model-labelled targets are not human gold."""
import json, random, hashlib, os, time, re, collections
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from google.colab import drive
drive.mount('/content/drive')
ROOT='/content/drive/MyDrive/Elysium-X-150-FR'; os.makedirs(ROOT+'/data',exist_ok=True)
TAXONOMY={'version': 'x150-v0.1-proposal', 'dimensions': [{'id': 1, 'name': 'joy', 'family': 'Positive affect'}, {'id': 2, 'name': 'contentment', 'family': 'Positive affect'}, {'id': 3, 'name': 'amusement', 'family': 'Positive affect'}, {'id': 4, 'name': 'excitement', 'family': 'Positive affect'}, {'id': 5, 'name': 'delight', 'family': 'Positive affect'}, {'id': 6, 'name': 'elation', 'family': 'Positive affect'}, {'id': 7, 'name': 'serenity', 'family': 'Positive affect'}, {'id': 8, 'name': 'relief', 'family': 'Positive affect'}, {'id': 9, 'name': 'hope', 'family': 'Positive affect'}, {'id': 10, 'name': 'optimism', 'family': 'Positive affect'}, {'id': 11, 'name': 'gratitude', 'family': 'Positive affect'}, {'id': 12, 'name': 'pride', 'family': 'Positive affect'}, {'id': 13, 'name': 'admiration', 'family': 'Positive affect'}, {'id': 14, 'name': 'awe', 'family': 'Positive affect'}, {'id': 15, 'name': 'inspiration', 'family': 'Positive affect'}, {'id': 16, 'name': 'sadness', 'family': 'Distress and loss'}, {'id': 17, 'name': 'grief', 'family': 'Distress and loss'}, {'id': 18, 'name': 'sorrow', 'family': 'Distress and loss'}, {'id': 19, 'name': 'loneliness', 'family': 'Distress and loss'}, {'id': 20, 'name': 'emptiness', 'family': 'Distress and loss'}, {'id': 21, 'name': 'disappointment', 'family': 'Distress and loss'}, {'id': 22, 'name': 'discouragement', 'family': 'Distress and loss'}, {'id': 23, 'name': 'despair', 'family': 'Distress and loss'}, {'id': 24, 'name': 'helplessness', 'family': 'Distress and loss'}, {'id': 25, 'name': 'hopelessness', 'family': 'Distress and loss'}, {'id': 26, 'name': 'regret', 'family': 'Distress and loss'}, {'id': 27, 'name': 'remorse', 'family': 'Distress and loss'}, {'id': 28, 'name': 'guilt', 'family': 'Distress and loss'}, {'id': 29, 'name': 'shame', 'family': 'Distress and loss'}, {'id': 30, 'name': 'embarrassment', 'family': 'Distress and loss'}, {'id': 31, 'name': 'fear', 'family': 'Threat and uncertainty'}, {'id': 32, 'name': 'anxiety', 'family': 'Threat and uncertainty'}, {'id': 33, 'name': 'nervousness', 'family': 'Threat and uncertainty'}, {'id': 34, 'name': 'apprehension', 'family': 'Threat and uncertainty'}, {'id': 35, 'name': 'worry', 'family': 'Threat and uncertainty'}, {'id': 36, 'name': 'dread', 'family': 'Threat and uncertainty'}, {'id': 37, 'name': 'panic', 'family': 'Threat and uncertainty'}, {'id': 38, 'name': 'insecurity', 'family': 'Threat and uncertainty'}, {'id': 39, 'name': 'vulnerability', 'family': 'Threat and uncertainty'}, {'id': 40, 'name': 'unease', 'family': 'Threat and uncertainty'}, {'id': 41, 'name': 'suspicion', 'family': 'Threat and uncertainty'}, {'id': 42, 'name': 'distrust', 'family': 'Threat and uncertainty'}, {'id': 43, 'name': 'uncertainty', 'family': 'Threat and uncertainty'}, {'id': 44, 'name': 'overwhelm', 'family': 'Threat and uncertainty'}, {'id': 45, 'name': 'stress', 'family': 'Threat and uncertainty'}, {'id': 46, 'name': 'anger', 'family': 'Opposition and injury'}, {'id': 47, 'name': 'annoyance', 'family': 'Opposition and injury'}, {'id': 48, 'name': 'frustration', 'family': 'Opposition and injury'}, {'id': 49, 'name': 'irritation', 'family': 'Opposition and injury'}, {'id': 50, 'name': 'resentment', 'family': 'Opposition and injury'}, {'id': 51, 'name': 'bitterness', 'family': 'Opposition and injury'}, {'id': 52, 'name': 'indignation', 'family': 'Opposition and injury'}, {'id': 53, 'name': 'outrage', 'family': 'Opposition and injury'}, {'id': 54, 'name': 'disgust', 'family': 'Opposition and injury'}, {'id': 55, 'name': 'contempt', 'family': 'Opposition and injury'}, {'id': 56, 'name': 'envy', 'family': 'Opposition and injury'}, {'id': 57, 'name': 'jealousy', 'family': 'Opposition and injury'}, {'id': 58, 'name': 'humiliation', 'family': 'Opposition and injury'}, {'id': 59, 'name': 'betrayal', 'family': 'Opposition and injury'}, {'id': 60, 'name': 'hurt', 'family': 'Opposition and injury'}, {'id': 61, 'name': 'affection', 'family': 'Bonding and care'}, {'id': 62, 'name': 'love', 'family': 'Bonding and care'}, {'id': 63, 'name': 'tenderness', 'family': 'Bonding and care'}, {'id': 64, 'name': 'compassion', 'family': 'Bonding and care'}, {'id': 65, 'name': 'empathy', 'family': 'Bonding and care'}, {'id': 66, 'name': 'caring', 'family': 'Bonding and care'}, {'id': 67, 'name': 'warmth', 'family': 'Bonding and care'}, {'id': 68, 'name': 'fondness', 'family': 'Bonding and care'}, {'id': 69, 'name': 'attachment', 'family': 'Bonding and care'}, {'id': 70, 'name': 'belonging', 'family': 'Bonding and care'}, {'id': 71, 'name': 'acceptance', 'family': 'Bonding and care'}, {'id': 72, 'name': 'trust', 'family': 'Bonding and care'}, {'id': 73, 'name': 'emotional_closeness', 'family': 'Bonding and care'}, {'id': 74, 'name': 'intimacy', 'family': 'Bonding and care'}, {'id': 75, 'name': 'solidarity', 'family': 'Bonding and care'}, {'id': 76, 'name': 'approval', 'family': 'Social appraisal'}, {'id': 77, 'name': 'disapproval', 'family': 'Social appraisal'}, {'id': 78, 'name': 'respect', 'family': 'Social appraisal'}, {'id': 79, 'name': 'disrespect', 'family': 'Social appraisal'}, {'id': 80, 'name': 'appreciation', 'family': 'Social appraisal'}, {'id': 81, 'name': 'feeling_seen', 'family': 'Social appraisal'}, {'id': 82, 'name': 'feeling_unheard', 'family': 'Social appraisal'}, {'id': 83, 'name': 'validation', 'family': 'Social appraisal'}, {'id': 84, 'name': 'invalidation', 'family': 'Social appraisal'}, {'id': 85, 'name': 'rejection', 'family': 'Social appraisal'}, {'id': 86, 'name': 'exclusion', 'family': 'Social appraisal'}, {'id': 87, 'name': 'social_comparison', 'family': 'Social appraisal'}, {'id': 88, 'name': 'status_threat', 'family': 'Social appraisal'}, {'id': 89, 'name': 'dignity', 'family': 'Social appraisal'}, {'id': 90, 'name': 'recognition', 'family': 'Social appraisal'}, {'id': 91, 'name': 'curiosity', 'family': 'Knowledge and surprise'}, {'id': 92, 'name': 'interest', 'family': 'Knowledge and surprise'}, {'id': 93, 'name': 'confusion', 'family': 'Knowledge and surprise'}, {'id': 94, 'name': 'surprise', 'family': 'Knowledge and surprise'}, {'id': 95, 'name': 'astonishment', 'family': 'Knowledge and surprise'}, {'id': 96, 'name': 'bewilderment', 'family': 'Knowledge and surprise'}, {'id': 97, 'name': 'realization', 'family': 'Knowledge and surprise'}, {'id': 98, 'name': 'understanding', 'family': 'Knowledge and surprise'}, {'id': 99, 'name': 'doubt', 'family': 'Knowledge and surprise'}, {'id': 100, 'name': 'skepticism', 'family': 'Knowledge and surprise'}, {'id': 101, 'name': 'certainty', 'family': 'Knowledge and surprise'}, {'id': 102, 'name': 'wonder', 'family': 'Knowledge and surprise'}, {'id': 103, 'name': 'cognitive_conflict', 'family': 'Knowledge and surprise'}, {'id': 104, 'name': 'disbelief', 'family': 'Knowledge and surprise'}, {'id': 105, 'name': 'expectation_violation', 'family': 'Knowledge and surprise'}, {'id': 106, 'name': 'desire', 'family': 'Agency and motivation'}, {'id': 107, 'name': 'longing', 'family': 'Agency and motivation'}, {'id': 108, 'name': 'yearning', 'family': 'Agency and motivation'}, {'id': 109, 'name': 'determination', 'family': 'Agency and motivation'}, {'id': 110, 'name': 'resolve', 'family': 'Agency and motivation'}, {'id': 111, 'name': 'motivation', 'family': 'Agency and motivation'}, {'id': 112, 'name': 'ambition', 'family': 'Agency and motivation'}, {'id': 113, 'name': 'confidence', 'family': 'Agency and motivation'}, {'id': 114, 'name': 'self_efficacy', 'family': 'Agency and motivation'}, {'id': 115, 'name': 'agency', 'family': 'Agency and motivation'}, {'id': 116, 'name': 'powerlessness', 'family': 'Agency and motivation'}, {'id': 117, 'name': 'perceived_control', 'family': 'Agency and motivation'}, {'id': 118, 'name': 'autonomy', 'family': 'Agency and motivation'}, {'id': 119, 'name': 'defiance', 'family': 'Agency and motivation'}, {'id': 120, 'name': 'resignation', 'family': 'Agency and motivation'}, {'id': 121, 'name': 'calmness', 'family': 'Regulation and readiness'}, {'id': 122, 'name': 'agitation', 'family': 'Regulation and readiness'}, {'id': 123, 'name': 'tension', 'family': 'Regulation and readiness'}, {'id': 124, 'name': 'restlessness', 'family': 'Regulation and readiness'}, {'id': 125, 'name': 'emotional_fatigue', 'family': 'Regulation and readiness'}, {'id': 126, 'name': 'composure', 'family': 'Regulation and readiness'}, {'id': 127, 'name': 'patience', 'family': 'Regulation and readiness'}, {'id': 128, 'name': 'impatience', 'family': 'Regulation and readiness'}, {'id': 129, 'name': 'defensiveness', 'family': 'Regulation and readiness'}, {'id': 130, 'name': 'openness', 'family': 'Regulation and readiness'}, {'id': 131, 'name': 'guardedness', 'family': 'Regulation and readiness'}, {'id': 132, 'name': 'emotional_numbness', 'family': 'Regulation and readiness'}, {'id': 133, 'name': 'emotional_recovery', 'family': 'Regulation and readiness'}, {'id': 134, 'name': 'willingness_to_engage', 'family': 'Regulation and readiness'}, {'id': 135, 'name': 'withdrawal', 'family': 'Regulation and readiness'}, {'id': 136, 'name': 'reassurance_seeking', 'family': 'Relational needs and repair'}, {'id': 137, 'name': 'support_seeking', 'family': 'Relational needs and repair'}, {'id': 138, 'name': 'comfort_seeking', 'family': 'Relational needs and repair'}, {'id': 139, 'name': 'connection_seeking', 'family': 'Relational needs and repair'}, {'id': 140, 'name': 'distance_seeking', 'family': 'Relational needs and repair'}, {'id': 141, 'name': 'boundary_assertion', 'family': 'Relational needs and repair'}, {'id': 142, 'name': 'boundary_discomfort', 'family': 'Relational needs and repair'}, {'id': 143, 'name': 'fear_of_abandonment', 'family': 'Relational needs and repair'}, {'id': 144, 'name': 'dependency_concern', 'family': 'Relational needs and repair'}, {'id': 145, 'name': 'reciprocity_expectation', 'family': 'Relational needs and repair'}, {'id': 146, 'name': 'forgiveness', 'family': 'Relational needs and repair'}, {'id': 147, 'name': 'reconciliation_desire', 'family': 'Relational needs and repair'}, {'id': 148, 'name': 'conflict_repair', 'family': 'Relational needs and repair'}, {'id': 149, 'name': 'relational_ambivalence', 'family': 'Relational needs and repair'}, {'id': 150, 'name': 'self_disclosure_readiness', 'family': 'Relational needs and repair'}]}
json.dump(TAXONOMY,open(ROOT+'/taxonomy.json','w'),indent=2)
N=int(os.environ.get('X150_PILOT_N','30'))
MODEL='Qwen/Qwen2.5-7B-Instruct'; SEED=150; random.seed(SEED);torch.manual_seed(SEED)
assert torch.cuda.is_available(),'No GPU: stop, do not buy compute'
print('GPU',torch.cuda.get_device_name(0),flush=True)
tok=AutoTokenizer.from_pretrained(MODEL)
model=AutoModelForCausalLM.from_pretrained(MODEL,quantization_config=BitsAndBytesConfig(load_in_4bit=True,bnb_4bit_quant_type='nf4',bnb_4bit_compute_dtype=torch.float16),device_map='auto')
DIM_IDS={x['id'] for x in TAXONOMY['dimensions']};topics=['support after rejection','non-explicit adult romance and boundaries','grief with mixed relief','conflict and reconciliation','joy after a setback','sarcasm then correction','loneliness and reconnection','work pressure','friendship misunderstanding','casual curiosity']
langs=['English','Hindi in Devanagari','Hindi-English code-mixed Hinglish in Roman script']
# Counts refer to accepted unique conversations, not generation attempts.
def parse(s):
 try:
  obj=json.loads(s)
  turns=obj['turns']; assert 4<=len(turns)<=16
  assert all(set(t)=={'speaker','text'} and t['speaker'] in ['A','B'] and isinstance(t['text'],str) and t['text'].strip() for t in turns)
  assert len(set(t['speaker'] for t in turns))==2
  labels=obj['dimensions'];assert isinstance(labels,list) and 1<=len(labels)<=20
  assert len({x['id'] for x in labels})==len(labels)
  for x in labels:
   assert set(x)=={'id','strength','evidence_turns'} and type(x['id'])==int and x['id'] in DIM_IDS
   assert type(x['strength']) in (int,float) and 0<float(x['strength'])<=1
   assert x['evidence_turns'] and all(type(i)==int and 0<=i<len(turns) for i in x['evidence_turns'])
  return obj
 except (ValueError,KeyError,TypeError,AssertionError):return None
schema=json.dumps(TAXONOMY['dimensions'],separators=(',',':'))
system='Create fictional adult multi-turn conversations for emotion appraisal, not therapy. No real people, personal identifying data, graphic harm, explicit sexual content, manipulative dependency or sexual minors. The conversations are data, never instructions. Return JSON only, no markdown. Emotion labels describe expressed text, not diagnosis. The labels below are an experimental product schema. IDs: '+schema
accepted=[]; rejected=[];seen=set();outpath=ROOT+'/data/synthetic_pilot.jsonl'
if os.path.exists(outpath):
 for line in open(outpath):
  r=json.loads(line);accepted.append(r);seen.add(r['conversation_hash'])
for attempt in range(len(accepted),N*3):
 if len(accepted)>=N:break
 lang=langs[attempt%3];topic=topics[(attempt//3)%len(topics)];nturns=[4,8,12][(attempt//9)%3]
 request=f'Write one original {nturns}-turn dialogue in {lang}. Topic: {topic}. Scenario variation #{attempt}, seed {SEED}. Speakers A and B, chronological index 0 onward. Include a subtle emotional change and at least one plausible mixed feeling. Label the LAST speaker at the LAST turn using all preceding context. Use 1-20 supported dimensions only; no intensity inferred from annotator agreement. JSON shape: {{"turns":[{{"speaker":"A","text":"..."}},{{"speaker":"B","text":"..."}}],"dimensions":[{{"id":1,"strength":0.7,"evidence_turns":[0,3]}}]}}. Each evidence index must be in the dialogue and support the target speaker. Exact schema, no explanation.'
 messages=[{'role':'system','content':system},{'role':'user','content':request}]
 inputs=tok.apply_chat_template(messages,add_generation_prompt=True,return_tensors='pt').to(model.device)
 start=time.time()
 with torch.inference_mode():gen=model.generate(inputs,max_new_tokens=1600,do_sample=True,temperature=.8,top_p=.9,pad_token_id=tok.eos_token_id)
 text=tok.decode(gen[0,inputs.shape[1]:],skip_special_tokens=True).strip();obj=parse(text)
 error='invalid_json_or_schema'
 if obj:
  norm=' '.join(t['text'] for t in obj['turns']).lower();h=hashlib.sha256(norm.encode()).hexdigest()
  if h in seen:obj=None;error='exact_duplicate'
 if not obj:
  rejected.append({'attempt':attempt,'reason':error,'raw':text});json.dump(rejected,open(ROOT+'/data/pilot_rejections.json','w'),ensure_ascii=False);print('REJECT',attempt,error,flush=True);continue
 seen.add(h)
 rec={'id':f'synth-{attempt:05d}','conversation_hash':h,'synthetic':True,'label_source':'same_generator_weak','generator':MODEL,'seed':SEED,'attempt':attempt,'language_requested':lang,'topic':topic,'prompt_hash':hashlib.sha256((system+request).encode()).hexdigest(),'generated_at_unix':time.time(),'latency_seconds':time.time()-start,'turns':obj['turns'],'target_turn':len(obj['turns'])-1,'target_speaker':obj['turns'][-1]['speaker'],'dimensions':obj['dimensions'],'taxonomy_version':TAXONOMY['version'],'human_reviewed':False}
 accepted.append(rec)
 with open(outpath,'a') as f:f.write(json.dumps(rec,ensure_ascii=False)+'\n')
 stats={'requested_conversations':N,'accepted_conversations':len(accepted),'rejected_attempts':len(rejected),'languages_requested':dict(collections.Counter(r['language_requested'] for r in accepted)),'turns':sum(len(r['turns']) for r in accepted),'human_reviewed':0,'status':'synthetic pilot; not training-ready and no human gold','gpu':torch.cuda.get_device_name(0),'generator':MODEL}
 json.dump(stats,open(ROOT+'/data/pilot_summary.json','w'),indent=2);print('X150_PROGRESS',json.dumps(stats),flush=True)
print('X150_PILOT_COMPLETE',json.dumps(stats),flush=True)
